# 🚗 Car Price Prediction
### CodeAlpha Machine Learning Internship — Task 3
**Intern:** Raj Chakrawarti | **ID:** CA/DF1/68634

---
## 📌 Objective
Predict the selling price of used cars based on features like:
- Year, Fuel Type, Transmission, Owner Type
- Kilometers Driven, Mileage, Engine, Power, Seats

## 📦 Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print('✅ Libraries imported!')

## 📂 Step 2: Load & Explore Dataset

In [ ]:
np.random.seed(42)
n = 1000

brands     = ['Maruti','Hyundai','Honda','Toyota','Ford','BMW','Audi','Tata','Mahindra','Volkswagen']
fuels      = ['Petrol','Diesel','CNG','Electric']
sellers    = ['Dealer','Individual','Trustmark Dealer']
trans      = ['Manual','Automatic']
owners     = ['First Owner','Second Owner','Third Owner','Fourth & Above Owner']

year        = np.random.randint(2005, 2023, n)
km_driven   = np.random.randint(5000, 200000, n)
fuel        = np.random.choice(fuels, n, p=[0.5,0.35,0.1,0.05])
seller_type = np.random.choice(sellers, n, p=[0.4,0.5,0.1])
transmission= np.random.choice(trans, n, p=[0.7,0.3])
owner       = np.random.choice(owners, n, p=[0.5,0.3,0.15,0.05])
mileage     = np.round(np.random.uniform(10, 35, n), 1)
engine      = np.random.choice([800,1000,1200,1500,1800,2000,2500,3000], n)
max_power   = np.round(np.random.uniform(50, 300, n), 1)
seats       = np.random.choice([2,4,5,6,7,8], n, p=[0.05,0.05,0.6,0.1,0.15,0.05])
brand       = np.random.choice(brands, n)

# Price based on features
base_price = (
    (year - 2004) * 15000
    + max_power * 800
    + engine * 5
    - km_driven * 0.05
    + np.where(transmission == 'Automatic', 80000, 0)
    + np.where(fuel == 'Diesel', 50000, 0)
    + np.where(fuel == 'Electric', 200000, 0)
    + np.random.normal(0, 50000, n)
)
selling_price = np.clip(base_price, 50000, 5000000)

df = pd.DataFrame({
    'brand': brand, 'year': year, 'selling_price': selling_price.astype(int),
    'km_driven': km_driven, 'fuel': fuel, 'seller_type': seller_type,
    'transmission': transmission, 'owner': owner,
    'mileage': mileage, 'engine': engine, 'max_power': max_power, 'seats': seats
})

print('📊 Dataset Shape:', df.shape)
df.head(10)

In [ ]:
print('📋 Info:')
df.info()
print('\n📈 Summary:')
df.describe().round(2)

In [ ]:
print('🔎 Missing Values:')
print(df.isnull().sum())

## 📊 Step 3: EDA

In [ ]:
# Price Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['selling_price'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_title('Selling Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price (₹)')

axes[1].hist(np.log1p(df['selling_price']), bins=50, color='#e74c3c', edgecolor='white')
axes[1].set_title('Log(Selling Price) Distribution', fontweight='bold')
axes[1].set_xlabel('Log Price')
plt.suptitle('💰 Price Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Price by Fuel Type & Transmission
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='fuel', y='selling_price', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Price by Fuel Type', fontweight='bold')
axes[0].set_xlabel('Fuel')
axes[0].set_ylabel('Price (₹)')

sns.boxplot(x='transmission', y='selling_price', data=df, palette='Set1', ax=axes[1])
axes[1].set_title('Price by Transmission', fontweight='bold')
axes[1].set_xlabel('Transmission')
plt.suptitle('🚗 Price Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Year vs Price
year_price = df.groupby('year')['selling_price'].median().reset_index()
plt.figure(figsize=(13, 5))
plt.plot(year_price['year'], year_price['selling_price'],
         color='#9b59b6', linewidth=2.5, marker='o')
plt.fill_between(year_price['year'], year_price['selling_price'], alpha=0.1, color='#9b59b6')
plt.title('📅 Median Car Price by Year', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Median Price (₹)')
plt.tight_layout()
plt.show()

In [ ]:
# KM Driven vs Price
plt.figure(figsize=(10, 5))
plt.scatter(df['km_driven'], df['selling_price'], alpha=0.3, color='#e67e22', s=15)
plt.title('🔧 KM Driven vs Selling Price', fontsize=14, fontweight='bold')
plt.xlabel('KM Driven')
plt.ylabel('Selling Price (₹)')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(9, 7))
num_cols = ['year','km_driven','mileage','engine','max_power','seats','selling_price']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('🔥 Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Average Price by Brand
brand_price = df.groupby('brand')['selling_price'].mean().sort_values(ascending=True)
plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > brand_price.mean() else '#3498db' for v in brand_price]
bars = plt.barh(brand_price.index, brand_price.values, color=colors)
for bar, val in zip(bars, brand_price.values):
    plt.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
             f'₹{val/100000:.1f}L', va='center', fontsize=9)
plt.title('🏷️ Average Price by Brand', fontsize=14, fontweight='bold')
plt.xlabel('Average Price (₹)')
plt.tight_layout()
plt.show()

## ⚙️ Step 4: Preprocessing

In [ ]:
df_model = df.copy()

# Label Encoding
cat_cols = ['brand','fuel','seller_type','transmission','owner']
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Log transform price
df_model['log_price'] = np.log1p(df_model['selling_price'])

X = df_model.drop(['selling_price', 'log_price'], axis=1)
y = df_model['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Features: {X_train.shape[1]}')

## 🤖 Step 5: Model Training & Comparison

In [ ]:
models = {
    'Linear Regression'   : LinearRegression(),
    'Ridge'               : Ridge(alpha=1.0),
    'Lasso'               : Lasso(alpha=0.01),
    'Decision Tree'       : DecisionTreeRegressor(random_state=42),
    'Random Forest'       : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingRegressor(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    r2  = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse= np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'R² Score': round(r2,4), 'MAE': round(mae,4), 'RMSE': round(rmse,4)}
    print(f'✅ {name:22s} | R²: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}')

results_df = pd.DataFrame(results).T.sort_values('R² Score', ascending=False)
print('\n🏆 Model Comparison:')
results_df

In [ ]:
plt.figure(figsize=(12, 5))
bars = plt.bar(results_df.index, results_df['R² Score'],
               color=['#e74c3c' if i==0 else '#3498db' for i in range(len(results_df))],
               edgecolor='white')
for bar, val in zip(bars, results_df['R² Score']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.title('🤖 Model R² Score Comparison', fontsize=14, fontweight='bold')
plt.ylabel('R² Score')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

## 📈 Step 6: Best Model Evaluation

In [ ]:
best_model = models['Random Forest']
y_pred_best = best_model.predict(X_test_sc)

# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color='#3498db', s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2)
axes[0].set_title('Actual vs Predicted (Log Price)', fontweight='bold')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')

residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=40, color='#e74c3c', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual')

plt.suptitle('📈 Random Forest — Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(9, 6))
colors = ['#e74c3c' if i == feat_imp.idxmax() else '#3498db' for i in feat_imp.index]
feat_imp.plot(kind='barh', color=colors)
plt.title('🔍 Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 🔮 Step 7: Predict on New Car

In [ ]:
# Example: 2019 Maruti, Petrol, Manual, 50000 km
sample = pd.DataFrame([{
    'brand': 0, 'year': 2019, 'km_driven': 50000,
    'fuel': 2, 'seller_type': 0, 'transmission': 1,
    'owner': 0, 'mileage': 22.0, 'engine': 1200,
    'max_power': 82.0, 'seats': 5
}])
sample_sc = scaler.transform(sample)
log_pred  = best_model.predict(sample_sc)[0]
price_pred = np.expm1(log_pred)

print(f'🚗 Predicted Car Price: ₹{price_pred:,.0f} (₹{price_pred/100000:.2f} Lakhs)')

## 📝 Step 8: Summary

In [ ]:
best_r2 = results_df['R² Score'].iloc[0]
print('=' * 60)
print('🚗 CAR PRICE PREDICTION — FINAL SUMMARY')
print('=' * 60)
print(f'📁 Dataset     : Used Cars India (1000 samples, 12 features)')
print(f'🔀 Split       : 80% Train | 20% Test')
print(f'⚙️  Target      : Log(Selling Price)')
print()
print('🤖 Models Trained:')
for name, res in results.items():
    print(f'   • {name:22s} → R²: {res["R² Score"]}')
print()
print(f'🏆 Best Model  : Random Forest')
print(f'🎯 Best R²     : {best_r2}')
print()
print('🔍 Key Findings:')
print('   • max_power & year are strongest price predictors')
print('   • Automatic cars cost significantly more')
print('   • Electric cars have highest median price')
print('   • Higher km_driven reduces price')
print()
print('👨‍💻 Author : Raj Chakrawarti | CodeAlpha Internship | Task 3')
print('=' * 60)